# BAN 612 Data Analytics
## Homework Assignment 4
### Market Expansion Using Census and External Data
- Hong Ngoc Dinh
- Nora Cook
- Hmwe Pyant Thin Chit

## Import Libraries

In [1]:
pip install tabulate


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
from tabulate import tabulate

## Census Data Collection

We use the 2024 ACS 5-Year Data Profile for California places.
The analysis includes population, income, education, and employment-related
variables that may be relevant for market expansion.

### Census Variables

- Total population (`DP05_0001E`): represents the overall size of the potential market.
- Median household income (`DP03_0062E`): provides an indication of household purchasing power.
- Bachelor's degree or higher (`DP02_0068PE`): represents the percentage of adults age 25+ with at least a bachelor's degree and can indicate the availability of an educated workforce.
- Labor force (`DP03_0002E`): market expansion may require new employees, so the size of the labor force helps indicate the availability of potential workers.
- Poverty rate (`DP03_0128PE`): a high poverty rate may indicate that a larger share of residents have limited purchasing power, which could affect demand for some products or services.
- Population 16 years and over (`DP03_0001E`): represents the population included in the ACS employment-status measures and provides a broad measure of the potential working-age population available in a market.

In [3]:
census_url = "https://api.census.gov/data/2024/acs/acs5/profile"

In [4]:
census_api_key = "5c133fc39b4396a1ecac3510d01b28e62217c988"

In [5]:
#Building the parameters
census_params = {
    "get": "NAME,DP05_0001E,DP03_0062E,DP02_0068PE,DP03_0001E,DP03_0002E,DP03_0128PE",
    "for": "place:*",
    "in": "state:06",
    "key": census_api_key
}

In [6]:
#Sending the API Request
census_response = requests.get(
    census_url,
    params=census_params,
    timeout=20
)

print(census_response.status_code)

200


In [7]:
census_json = census_response.json()

census_json[:3]

[['NAME',
  'DP05_0001E',
  'DP03_0062E',
  'DP02_0068PE',
  'DP03_0001E',
  'DP03_0002E',
  'DP03_0128PE',
  'state',
  'place'],
 ['Acalanes Ridge CDP, California',
  '1006',
  '181094',
  '70.2',
  '954',
  '480',
  '1.5',
  '06',
  '00135'],
 ['Acampo CDP, California',
  '234',
  '-666666666',
  '21.8',
  '234',
  '110',
  '31.2',
  '06',
  '00156']]

In [8]:
census_df = pd.DataFrame(
    census_json[1:],
    columns=census_json[0]
)

census_df.head()

,NAME,DP05_0001E,DP03_0062E,DP02_0068PE,DP03_0001E,DP03_0002E,DP03_0128PE,state,place
0,"Acalanes Ridge CDP, California",1006,181094,70.2,954,480,1.5,06,00135
1,"Acampo CDP, California",234,-666666666,21.8,234,110,31.2,06,00156
2,"Acton CDP, California",7880,113000,29.9,6402,3626,13.1,06,00212
3,"Adelanto city, California",37964,68685,7.7,26452,14790,17.2,06,00296
4,"Adin CDP, California",153,29375,31.3,153,68,4.6,06,00310


In [9]:
#Renaming the columns
census_df = census_df.rename(columns={
    "NAME": "place_name",
    "DP05_0001E": "population",
    "DP03_0062E": "median_household_income",
    "DP02_0068PE": "bachelors_or_higher_pct",
    "DP03_0001E": "population_16_plus",
    "DP03_0002E": "labor_force",
    "DP03_0128PE": "poverty_rate"
})

census_df.head()

,place_name,population,median_household_income,bachelors_or_higher_pct,population_16_plus,labor_force,poverty_rate,state,place
0,"Acalanes Ridge CDP, California",1006,181094,70.2,954,480,1.5,06,00135
1,"Acampo CDP, California",234,-666666666,21.8,234,110,31.2,06,00156
2,"Acton CDP, California",7880,113000,29.9,6402,3626,13.1,06,00212
3,"Adelanto city, California",37964,68685,7.7,26452,14790,17.2,06,00296
4,"Adin CDP, California",153,29375,31.3,153,68,4.6,06,00310


In [10]:
census_df.dtypes

place_name                 str
population                 str
median_household_income    str
bachelors_or_higher_pct    str
population_16_plus         str
labor_force                str
poverty_rate               str
state                      str
place                      str
dtype: object

In [11]:
#Converting analysis columns to numeric
numeric_cols = [
    "population",
    "median_household_income",
    "bachelors_or_higher_pct",
    "population_16_plus",
    "labor_force",
    "poverty_rate"
]

census_df[numeric_cols] = census_df[numeric_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

In [12]:
census_df.dtypes

place_name                     str
population                   int64
median_household_income      int64
bachelors_or_higher_pct    float64
population_16_plus           int64
labor_force                  int64
poverty_rate               float64
state                          str
place                          str
dtype: object

In [13]:
census_df[numeric_cols].describe()

,population,median_household_income,bachelors_or_higher_pct,population_16_plus,labor_force,poverty_rate
count,1.618000e+03,1.618000e+03,1.618000e+03,1.618000e+03,1.618000e+03,1.618000e+03
mean,2.318506e+04,-1.000382e+08,-1.977747e+07,1.869546e+04,1.199367e+04,-2.101358e+07
std,1.152580e+05,2.382782e+08,1.131449e+08,9.536056e+04,6.353768e+04,1.165156e+08
min,0.000000e+00,-6.666667e+08,-6.666667e+08,0.000000e+00,0.000000e+00,-6.666667e+08
25%,4.982500e+02,5.125000e+04,1.202500e+01,4.170000e+02,2.222500e+02,5.100000e+00
50%,2.810500e+03,8.087700e+04,2.480000e+01,2.293000e+03,1.291500e+03,9.600000e+00
75%,1.483475e+04,1.134768e+05,4.377500e+01,1.191275e+04,7.001000e+03,1.730000e+01
max,3.857263e+06,2.500010e+05,1.000000e+02,3.204204e+06,2.130571e+06,1.000000e+02


In [14]:
#Indentifying invalid values
census_df[
    (census_df["median_household_income"] < 0) |
    (census_df["bachelors_or_higher_pct"] < 0) |
    (census_df["poverty_rate"] < 0)
][[
    "place_name",
    "median_household_income",
    "bachelors_or_higher_pct",
    "poverty_rate"
]].head(10)

,place_name,median_household_income,bachelors_or_higher_pct,poverty_rate
1,"Acampo CDP, California",-666666666,21.8,31.2
13,"Albion CDP, California",-666666666,18.8,39.5
14,"Alderpoint CDP, California",-666666666,0.0,100.0
18,"Alleghany CDP, California",-666666666,0.0,0.0
21,"Almanor CDP, California",-666666666,-666666666.0,-666666666.0
28,"Alta Sierra CDP (Kern County), California",-666666666,100.0,0.0
74,"Baker CDP, California",-666666666,0.0,28.4
90,"Bear Valley CDP (Alpine County), California",-666666666,80.8,0.0
91,"Bear Valley CDP (Mariposa County), California",-666666666,10.4,20.1
94,"Beckwourth CDP, California",-666666666,54.6,0.0


In [15]:
clean_census_df = census_df[
    (census_df["population"] > 0) &
    (census_df["median_household_income"] >= 0) &
    (census_df["bachelors_or_higher_pct"].between(0, 100)) &
    (census_df["population_16_plus"] >= 0) &
    (census_df["labor_force"] >= 0) &
    (census_df["labor_force"] <= census_df["population_16_plus"]) &
    (census_df["poverty_rate"].between(0, 100))
].copy()

print("Original rows:", len(census_df))
print("Clean rows:", len(clean_census_df))
print("Removed rows:", len(census_df) - len(clean_census_df))

Original rows: 1618
Clean rows: 1375
Removed rows: 243


In [71]:
cities_df = clean_census_df[
    clean_census_df["place_name"].str.contains(" city, California", na=False)
].copy()

print("Number of California cities:", len(cities_df))

Number of California cities: 461


In [19]:
original_cities = ["Fremont city, California",
                   "Sunnyvale city, California",
                   "Irvine city, California"]

cities_df[
    cities_df["place_name"].isin(original_cities)
]

,place_name,population,median_household_income,bachelors_or_higher_pct,population_16_plus,labor_force,poverty_rate,state,place
529,"Fremont city, California",228295,181506,63.9,183206,124445,5.2,06,26000
675,"Irvine city, California",311690,136719,70.6,251536,169183,10.8,06,36770
1414,"Sunnyvale city, California",154236,186170,69.3,127071,92133,5.8,06,77000


### Additional Census Variables for Market Expansion

In addition to the Census variables used in Homework 3, three additional variables were included:

**1. Population 16 years and over (`DP03_0001E`)**  
This variable represents the population age 16 and over included in the ACS employment-status measures. It is useful for market expansion because it provides information about the size of the population that could potentially participate in the labor market.

**2. Labor force (`DP03_0002E`)**  
The labor force includes people age 16 and over who are working or actively looking for work. Market expansion may require hiring new employees, so a larger labor force can indicate a larger pool of potential workers.

**3. Poverty rate (`DP03_0128PE`)**  
The poverty rate represents the percentage of all people whose income is below the poverty level. A higher poverty rate may indicate that a larger share of residents have limited purchasing power, which could affect demand for some products or services.

## External Zillow Home Value Data

Zillow Home Value Index (ZHVI) city-level data is used as an external dataset to provide information about typical home values and housing costs across cities.

In [20]:
zillow_df = pd.read_csv(
    "City_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
)

zillow_df.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,CountyName,2000-01-31,2000-02-29,...,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31
0,6181,0,New York,city,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",Queens County,244683.142667,245985.301407,...,807979.551349,811893.147844,815077.202854,818580.166556,823656.034498,828850.128461,831538.590696,831817.219139,831074.675460,832934.229552
1,12447,1,Los Angeles,city,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",Los Angeles County,224633.147635,224973.266675,...,954999.508359,958575.242429,962298.766268,963529.970421,962349.706885,958580.598954,954269.253830,950650.432347,947629.042082,946268.116724
2,39051,2,Houston,city,TX,TX,"Houston-The Woodlands-Sugar Land, TX",Harris County,105097.717383,105071.014816,...,268246.608985,267963.464323,268012.966979,267952.067983,267639.136728,267026.669107,266254.512780,265278.975912,264288.030649,263562.906937
3,17426,3,Chicago,city,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",Cook County,154017.059585,154058.208305,...,326083.855405,327418.056818,328612.880047,329630.539539,331097.574087,332497.501158,333762.932398,334595.584827,335886.618187,337992.649416
4,6915,4,San Antonio,city,TX,TX,"San Antonio-New Braunfels, TX",Bexar County,101410.038625,101504.186188,...,254153.924770,253771.068706,253478.811813,253178.256298,252938.406442,252645.977874,252205.871717,251432.137002,250492.790292,249688.965116


In [21]:
zillow_df.columns[-10:]

Index(['2025-10-31', '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28',
       '2026-03-31', '2026-04-30', '2026-05-31', '2026-06-30', '2026-07-31'],
      dtype='str')

### Zillow Home Value Data

The Zillow Home Value Index (ZHVI) city-level dataset contains monthly typical home values. 
The latest available month in the downloaded dataset is August 2026, so the August 2026 ZHVI 
is used as the current home-value measure for the market-expansion analysis.

In [22]:
zillow_ca = zillow_df[
    zillow_df["State"] == "CA"
].copy()

zillow_ca.shape

(959, 327)

In [23]:
zillow_ca = zillow_ca[
    ["RegionName", "State", "CountyName", "2026-07-31"]
].copy()

zillow_ca.head()

,RegionName,State,CountyName,2026-07-31
1,Los Angeles,CA,Los Angeles County,9.462681e+05
8,San Diego,CA,San Diego County,9.946821e+05
11,San Jose,CA,Santa Clara County,1.391204e+06
17,San Francisco,CA,San Francisco County,1.416278e+06
28,Sacramento,CA,Sacramento County,4.804876e+05


In [24]:
zillow_ca = zillow_ca.rename(columns={
    "RegionName": "city_name",
    "State": "state",
    "CountyName": "county_name",
    "2026-07-31": "zhvi_2026_07"
})

zillow_ca.head()

,city_name,state,county_name,zhvi_2026_07
1,Los Angeles,CA,Los Angeles County,9.462681e+05
8,San Diego,CA,San Diego County,9.946821e+05
11,San Jose,CA,Santa Clara County,1.391204e+06
17,San Francisco,CA,San Francisco County,1.416278e+06
28,Sacramento,CA,Sacramento County,4.804876e+05


In [25]:
cities_df["city_name"] = (
    cities_df["place_name"]
    .str.replace(" city, California", "", regex=False)
    .str.strip()
)

cities_df[["place_name", "city_name"]].head()

,place_name,city_name
3,"Adelanto city, California",Adelanto
5,"Agoura Hills city, California",Agoura Hills
10,"Alameda city, California",Alameda
12,"Albany city, California",Albany
15,"Alhambra city, California",Alhambra


In [26]:
zillow_ca[
    zillow_ca["city_name"].isin(["Fremont", "Sunnyvale", "Irvine"])
]

,city_name,state,county_name,zhvi_2026_07
84,Irvine,CA,Orange County,1.517431e+06
123,Fremont,CA,Alameda County,1.471931e+06
231,Sunnyvale,CA,Santa Clara County,2.050814e+06


In [27]:
# Merge Cencus and Zillow data
merged_zillow = cities_df.merge(
    zillow_ca,
    on="city_name",
    how="inner",
    suffixes=("_census", "_zillow")
)

merged_zillow.shape

(450, 13)

In [28]:
unmatched_cities = cities_df[
    ~cities_df["city_name"].isin(zillow_ca["city_name"])
]

unmatched_cities[["place_name", "city_name"]]

,place_name,city_name
40,"Angels city, California",Angels
457,"El Paso de Robles (Paso Robles) city, California",El Paso de Robles (Paso Robles)
668,"Industry city, California",Industry
722,"La Cañada Flintridge city, California",La Cañada Flintridge
866,"McFarland city, California",McFarland
958,"Monte Sereno city, California",Monte Sereno
1248,"St. Helena city, California",St. Helena
1262,"San Buenaventura (Ventura) city, California",San Buenaventura (Ventura)
1265,"Sand City city, California",Sand City
1510,"Vernon city, California",Vernon


In [29]:
zillow_ca[
    zillow_ca["city_name"].str.contains(
        "Robles|Ventura|Angels|Helena|Canada|McFarland|Monte Sereno|Industry|McFarland|Monte Sereno|Sand City|Vernon|Westmorland",
        case=False,
        na=False
    )
][["city_name", "county_name", "zhvi_2026_07"]]

,city_name,county_name,zhvi_2026_07
376,Ventura,Ventura County,8.982407e+05
1370,Paso Robles,San Luis Obispo County,7.775307e+05
2962,La Canada Flintridge,Los Angeles County,2.509075e+06
6187,Saint Helena,Napa County,1.642736e+06
7792,Angels Camp,Calaveras County,4.584459e+05


In [30]:
# Standardizing the 5 verified names
cities_df["merge_city_name"] = cities_df["city_name"]

name_corrections = {
    "Angels": "Angels Camp",
    "El Paso de Robles (Paso Robles)": "Paso Robles",
    "La Cañada Flintridge": "La Canada Flintridge",
    "St. Helena": "Saint Helena",
    "San Buenaventura (Ventura)": "Ventura"
}

cities_df["merge_city_name"] = (
    cities_df["merge_city_name"].replace(name_corrections)
)

In [31]:
zillow_ca["merge_city_name"] = zillow_ca["city_name"]

In [32]:
merged_zillow = cities_df.merge(
    zillow_ca,
    on="merge_city_name",
    how="inner",
    suffixes=("_census", "_zillow")
)

merged_zillow.shape

(455, 15)

In [33]:
merged_zillow["zhvi_2026_07"].isna().sum()

np.int64(0)

In [34]:
merged_zillow

,place_name,population,median_household_income,bachelors_or_higher_pct,population_16_plus,labor_force,poverty_rate,state_census,place,city_name_census,merge_city_name,city_name_zillow,state_zillow,county_name,zhvi_2026_07
0,"Adelanto city, California",37964,68685,7.7,26452,14790,17.2,06,00296,Adelanto,Adelanto,Adelanto,CA,San Bernardino County,4.022460e+05
1,"Agoura Hills city, California",19721,171302,67.5,16071,10086,6.5,06,00394,Agoura Hills,Agoura Hills,Agoura Hills,CA,Los Angeles County,1.242040e+06
2,"Alameda city, California",77238,137697,61.5,61948,41480,7.2,06,00562,Alameda,Alameda,Alameda,CA,Alameda County,1.156096e+06
3,"Albany city, California",19414,144107,77.8,15817,10526,6.8,06,00674,Albany,Albany,Albany,CA,Alameda County,1.249885e+06
4,"Alhambra city, California",81022,88024,40.8,68755,43797,12.6,06,00884,Alhambra,Alhambra,Alhambra,CA,Los Angeles County,9.276946e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450,"Yorba Linda city, California",67170,157582,56.2,53756,33297,5.8,06,86832,Yorba Linda,Yorba Linda,Yorba Linda,CA,Orange County,1.407941e+06
451,"Yountville city, California",3348,80342,41.6,3236,1410,14.7,06,86930,Yountville,Yountville,Yountville,CA,Napa County,1.306018e+06
452,"Yreka city, California",7820,43982,19.0,6199,3273,17.0,06,86944,Yreka,Yreka,Yreka,CA,Siskiyou County,2.589152e+05
453,"Yuba City city, California",69471,75477,20.4,54167,33066,14.3,06,86972,Yuba City,Yuba City,Yuba City,CA,Sutter County,4.451450e+05


### Geographic Cleaning and Merge

The Census and Zillow datasets used different naming conventions for some California cities. 
City names were standardized before merging. For example, Census listed 
"San Buenaventura (Ventura)" while Zillow listed "Ventura," and Census listed 
"St. Helena" while Zillow listed "Saint Helena."

The datasets were merged using the standardized city name. Of the 461 California cities 
in the cleaned Census dataset, 455 matched with Zillow, resulting in a match rate of 
approximately 98.7%. Six cities did not have a verified Zillow match and were not included 
in the merged dataset.

In [35]:
merged_zillow["labor_force_participation_rate"] = (
    merged_zillow["labor_force"] /
    merged_zillow["population_16_plus"]
) * 100

print(merged_zillow["labor_force_participation_rate"])

0      55.912596
1      62.759007
2      66.959385
3      66.548650
4      63.700095
         ...    
450    61.940993
451    43.572311
452    52.798839
453    61.044547
454    62.201208
Name: labor_force_participation_rate, Length: 455, dtype: float64


In [36]:
top_3_lf = merged_zillow[
    merged_zillow["city_name_census"].isin(["Fremont", "Sunnyvale", "Irvine"])
][[
    "city_name_census",
    "population_16_plus",
    "labor_force",
    "labor_force_participation_rate"
]]
top_3_lf

,city_name_census,population_16_plus,labor_force,labor_force_participation_rate
144,Fremont,183206,124445,67.926269
186,Irvine,251536,169183,67.259955
404,Sunnyvale,127071,92133,72.505135


A higher labor force participation rate means more people are working or looking for work. This can be good for market expansion because it shows that more people are economically active and may have spending power.

In [37]:
merged_zillow["home_value_to_income_ratio"] = (
    merged_zillow["zhvi_2026_07"] /
    merged_zillow["median_household_income"]
)


In [38]:
top_3_home = merged_zillow[
    merged_zillow["city_name_census"].isin(["Fremont", "Sunnyvale", "Irvine"])
][[
    "city_name_census",
    "median_household_income",
    "zhvi_2026_07",
    "home_value_to_income_ratio"
]]
top_3_home

,city_name_census,median_household_income,zhvi_2026_07,home_value_to_income_ratio
144,Fremont,181506,1.471931e+06,8.109547
186,Irvine,136719,1.517431e+06,11.098905
404,Sunnyvale,186170,2.050814e+06,11.015813


Home value-to-income ratio compares typical home value with median household income. A lower ratio may indicate that housing is more affordable compared with local income, which can be important when considering cost of living for market expansion.

In [39]:
merged_zillow["non_poverty_rate"] = (
    100 - merged_zillow["poverty_rate"]
)

In [40]:
top_3_non_poverty = merged_zillow[
    merged_zillow["city_name_census"].isin(["Fremont", "Sunnyvale", "Irvine"])
][[
    "city_name_census",
    "poverty_rate",
    "non_poverty_rate"
]]
top_3_non_poverty

,city_name_census,poverty_rate,non_poverty_rate
144,Fremont,5.2,94.8
186,Irvine,10.8,89.2
404,Sunnyvale,5.8,94.2


A higher non-poverty rate means a larger percentage of the population is above the poverty level. This may indicate stronger purchasing power for products and services.

In [41]:
merged_zillow["labor_force_to_population_rate"] = (
    merged_zillow["labor_force"] /
    merged_zillow["population"]
) * 100

In [42]:
merged_zillow[
    merged_zillow["city_name_census"].isin(["Fremont", "Sunnyvale", "Irvine"])
][[
    "city_name_census",
    "population",
    "labor_force",
    "labor_force_to_population_rate"
]]

,city_name_census,population,labor_force,labor_force_to_population_rate
144,Fremont,228295,124445,54.510611
186,Irvine,311690,169183,54.279252
404,Sunnyvale,154236,92133,59.735081


This measure shows how large the labor force is compared with the city's total population. A higher percentage may indicate a larger economically active population, which can be useful for market expansion.

### Summary of Derived Variables

1. **Labor Force Participation Rate** = Labor Force ÷ Population 16+ × 100  
   - Shows the percentage of people age 16 and over who are working or looking for work.

2. **Home Value-to-Income Ratio** = Typical Home Value ÷ Median Household Income  
   - Shows housing cost relative to local income. A lower ratio may indicate better housing affordability.

3. **Non-Poverty Rate** = 100 − Poverty Rate  
   - Shows the percentage of people who are not below the poverty level. A higher rate may indicate stronger purchasing power.

4. **Labor Force-to-Population Rate** = Labor Force ÷ Total Population × 100  
   - Shows how large the labor force is compared with the city's total population.

**Difference between Variables 1 and 4:**  
These two measures are similar, but they use different populations as the denominator. Labor force participation rate uses only the population age 16 and over, while labor force-to-population rate uses the city's total population, including children. Therefore, they provide slightly different views of the city's economically active population.

## Re-evaluating the Original Three Cities

In [43]:
original_three = merged_zillow[
    merged_zillow["city_name_census"].isin(["Fremont", "Sunnyvale", "Irvine"])
][[
    "city_name_census",
    "population",
    "median_household_income",
    "bachelors_or_higher_pct",
    "poverty_rate",
    "zhvi_2026_07",
    "labor_force_participation_rate",
    "home_value_to_income_ratio",
    "non_poverty_rate",
    "labor_force_to_population_rate"
]]

original_three

,city_name_census,population,median_household_income,bachelors_or_higher_pct,poverty_rate,zhvi_2026_07,labor_force_participation_rate,home_value_to_income_ratio,non_poverty_rate,labor_force_to_population_rate
144,Fremont,228295,181506,63.9,5.2,1.471931e+06,67.926269,8.109547,94.8,54.510611
186,Irvine,311690,136719,70.6,10.8,1.517431e+06,67.259955,11.098905,89.2,54.279252
404,Sunnyvale,154236,186170,69.3,5.8,2.050814e+06,72.505135,11.015813,94.2,59.735081


**Fremont:**  
Fremont still looks attractive because it has the highest non-poverty rate at 94.8%, which may indicate stronger purchasing power. However, its labor-force participation rate is 67.9%, which is lower than Sunnyvale's 72.5%.

**Irvine:**  
Irvine still has the highest percentage of people with a bachelor's degree or higher at 70.6%, which indicates a highly educated population. However, the new data shows that Irvine has the highest poverty rate at 10.8% among the three cities, which may be a concern for purchasing power.

**Sunnyvale:**  
Sunnyvale has the highest labor-force-to-population rate at 59.7% and the highest labor-force participation rate at 72.5%, showing a relatively large economically active population. However, its typical home value is about $1.99 million, the highest among the three cities, which may indicate a high cost of living.

Overall, the three cities still show strengths for market expansion, but the additional Census and Zillow data reveal new trade-offs that were not considered in Homework 3.

## Identifying two alternative California cities that look attractive under the expanded dataset

In [44]:
#Excluded original three cities
alternative_cities = merged_zillow[
    ~merged_zillow["city_name_census"].isin(
        ["Fremont", "Sunnyvale", "Irvine"]
    )
].copy()

In [45]:
alternative_cities.shape

(452, 19)

In [46]:
alternative_cities[[
    "city_name_census",
    "population",
    "median_household_income",
    "bachelors_or_higher_pct",
    "poverty_rate",
    "zhvi_2026_07",
    "labor_force_participation_rate",
    "home_value_to_income_ratio",
    "non_poverty_rate",
    "labor_force_to_population_rate"]].describe()


,population,median_household_income,bachelors_or_higher_pct,poverty_rate,zhvi_2026_07,labor_force_participation_rate,home_value_to_income_ratio,non_poverty_rate,labor_force_to_population_rate
count,4.520000e+02,452.000000,452.000000,452.000000,4.520000e+02,452.000000,452.000000,452.000000,452.000000
mean,7.030589e+04,103479.878319,34.935619,11.988496,9.248033e+05,61.316405,8.329149,88.011504,49.252677
std,2.096566e+05,43403.140206,20.580346,6.410551,7.170575e+05,7.674840,3.227881,6.410551,6.721714
min,1.740000e+02,35540.000000,1.500000,2.200000,1.401465e+05,21.497444,2.880383,59.000000,21.334759
25%,1.232350e+04,73656.750000,17.025000,7.200000,4.718680e+05,58.504775,6.049875,84.375000,45.508858
50%,3.231900e+04,93904.500000,30.800000,10.550000,7.360174e+05,62.764911,7.811238,89.450000,50.386691
75%,7.639750e+04,122153.250000,49.100000,15.625000,1.101193e+06,66.175526,9.743413,92.800000,53.690358
max,3.857263e+06,250001.000000,88.400000,41.000000,5.314351e+06,79.010239,27.928619,97.800000,74.011008


In [47]:
strong_alternatives = alternative_cities[
    (alternative_cities["median_household_income"] >= 122153.25) &
    (alternative_cities["poverty_rate"] <= 7.2) &
    (alternative_cities["labor_force_participation_rate"] >= 66.175526) &
    (alternative_cities["home_value_to_income_ratio"] <= 5.995667)
]

strong_alternatives[[
    "city_name_census",
    "population",
    "median_household_income",
    "bachelors_or_higher_pct",
    "poverty_rate",
    "zhvi_2026_07",
    "labor_force_participation_rate",
    "home_value_to_income_ratio",
    "non_poverty_rate",
    "labor_force_to_population_rate"
]]

,city_name_census,population,median_household_income,bachelors_or_higher_pct,poverty_rate,zhvi_2026_07,labor_force_participation_rate,home_value_to_income_ratio,non_poverty_rate,labor_force_to_population_rate
113,Dublin,70803,214385,69.8,4.5,1.250804e+06,70.748609,5.834384,95.5,54.939762
116,Eastvale,70633,160069,43.4,4.8,9.425420e+05,69.805864,5.888348,95.2,52.281511
277,Oakley,45183,137677,24.3,6.1,6.606489e+05,68.558145,4.798542,93.9,54.175243


In [48]:
strong_alternatives[[
    "city_name_census",
    "population",
    "median_household_income",
    "bachelors_or_higher_pct",
    "poverty_rate",
    "zhvi_2026_07",
    "labor_force_participation_rate",
    "home_value_to_income_ratio",
    "non_poverty_rate",
    "labor_force_to_population_rate"]]


,city_name_census,population,median_household_income,bachelors_or_higher_pct,poverty_rate,zhvi_2026_07,labor_force_participation_rate,home_value_to_income_ratio,non_poverty_rate,labor_force_to_population_rate
113,Dublin,70803,214385,69.8,4.5,1.250804e+06,70.748609,5.834384,95.5,54.939762
116,Eastvale,70633,160069,43.4,4.8,9.425420e+05,69.805864,5.888348,95.2,52.281511
277,Oakley,45183,137677,24.3,6.1,6.606489e+05,68.558145,4.798542,93.9,54.175243


### Alternative Cities

**Selection Method:**  
We first filtered the cities using the top 25% for median household income and labor-force participation rate, and the lowest 25% for poverty rate and home-value-to-income ratio. Three cities met all four conditions: Dublin, Eastvale, and Oakley. I then compared these cities using population, education, and the other available variables to select two alternatives.

**Dublin:**  
Dublin has a population of 70,803, a high median household income of $214,385, and a low poverty rate of 4.5%. It also has a high labor-force participation rate of 70.7%. These factors suggest a strong potential customer base and labor market.

**Eastvale:**  
Eastvale has a population of 70,633, a median household income of $160,069, and a low poverty rate of 4.8%. Its labor-force participation rate is also high at 69.8%. These characteristics make Eastvale another potential location for market expansion.

**Oakley:**  
Dublin and Eastvale were selected over Oakley because they have larger populations and lower poverty rates. Oakley has the lowest home-value-to-income ratio, which is a strength, but it has a smaller population of 45,183 and a higher poverty rate of 6.1%.

## Overall Attractivness score

**Economic Stability:**
non_poverty_rate (35%): 
Why it matters: This serves as the foundation of city health. A high non-poverty rate correlates strongly with lower crime, better funding for public infrastructure, stronger local businesses, and overall community well-being. It prevents a city with high wages but extreme wealth disparity from scoring artificially high.

**Human Capital:** bachelors_or_higher_pct (25%)
Why it matters: A highly educated workforce attracts top-tier employers, tech hubs, and high-paying industries. This metric acts as a proxy for cultural amenities, innovation, future economic resilience, and long-term wage growth potential.

**Affordability:**
home_value_to_income_ratio (20%)
Why it matters: A city can have great jobs, but it loses its attractiveness if everyday workers are priced out of housing.Important Adjustment: Unlike the other metrics, a lower home value-to-income ratio is more attractive. You must invert this metric (e.g., using a formula where a lower ratio yields a higher score closer to 100) before running the weighted average.

**Economic Engagement:**
labor_force_to_population_rate (20%)
Why it matters: This captures the active, working-age demographic of the city. A high rate indicates a vibrant, dynamic job market with plenty of active opportunities. It penalizes cities that might have wealthy retirees or passive income earners but lack a thriving local labor economy.

In [49]:
max_ratio = merged_zillow['home_value_to_income_ratio'].max()
min_ratio = merged_zillow['home_value_to_income_ratio'].min()

merged_zillow['affordability_score'] = (
    (max_ratio - merged_zillow['home_value_to_income_ratio']) / (max_ratio - min_ratio)
) * 100

In [50]:
merged_zillow

,place_name,population,median_household_income,bachelors_or_higher_pct,population_16_plus,labor_force,poverty_rate,state_census,place,city_name_census,merge_city_name,city_name_zillow,state_zillow,county_name,zhvi_2026_07,labor_force_participation_rate,home_value_to_income_ratio,non_poverty_rate,labor_force_to_population_rate,affordability_score
0,"Adelanto city, California",37964,68685,7.7,26452,14790,17.2,06,00296,Adelanto,Adelanto,Adelanto,CA,San Bernardino County,4.022460e+05,55.912596,5.856388,82.8,38.957960,88.118904
1,"Agoura Hills city, California",19721,171302,67.5,16071,10086,6.5,06,00394,Agoura Hills,Agoura Hills,Agoura Hills,CA,Los Angeles County,1.242040e+06,62.759007,7.250585,93.5,51.143451,82.552856
2,"Alameda city, California",77238,137697,61.5,61948,41480,7.2,06,00562,Alameda,Alameda,Alameda,CA,Alameda County,1.156096e+06,66.959385,8.395942,92.8,53.704135,77.980251
3,"Albany city, California",19414,144107,77.8,15817,10526,6.8,06,00674,Albany,Albany,Albany,CA,Alameda County,1.249885e+06,66.548650,8.673314,93.2,54.218605,76.872897
4,"Alhambra city, California",81022,88024,40.8,68755,43797,12.6,06,00884,Alhambra,Alhambra,Alhambra,CA,Los Angeles County,9.276946e+05,63.700095,10.539110,87.4,54.055689,69.424087
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450,"Yorba Linda city, California",67170,157582,56.2,53756,33297,5.8,06,86832,Yorba Linda,Yorba Linda,Yorba Linda,CA,Orange County,1.407941e+06,61.940993,8.934654,94.2,49.571237,75.829553
451,"Yountville city, California",3348,80342,41.6,3236,1410,14.7,06,86930,Yountville,Yountville,Yountville,CA,Napa County,1.306018e+06,43.572311,16.255730,85.3,42.114695,46.601639
452,"Yreka city, California",7820,43982,19.0,6199,3273,17.0,06,86944,Yreka,Yreka,Yreka,CA,Siskiyou County,2.589152e+05,52.798839,5.886844,83.0,41.854220,87.997313
453,"Yuba City city, California",69471,75477,20.4,54167,33066,14.3,06,86972,Yuba City,Yuba City,Yuba City,CA,Sutter County,4.451450e+05,61.044547,5.897756,85.7,47.596839,87.953748


In [51]:
merged_zillow['attractiveness_score'] = (
    (merged_zillow['non_poverty_rate'] * 0.35) +
    (merged_zillow['bachelors_or_higher_pct'] * 0.25) +
    (merged_zillow['labor_force_to_population_rate'] * 0.20) +
    (merged_zillow['affordability_score'] * 0.20)  # Swap with _absolute if preferred
)

# Sort by the most attractive city
merged_zillow_ranked = merged_zillow.sort_values(by='attractiveness_score', ascending=False)
print(merged_zillow_ranked[['place_name', 'home_value_to_income_ratio', 'attractiveness_score']].head(10))


                         place_name  home_value_to_income_ratio  \
124     Emeryville city, California                    3.752109   
283         Orinda city, California                    7.951164   
380      Sausalito city, California                    8.629598   
113         Dublin city, California                    5.834384   
195      Lafayette city, California                    8.504312   
301       Piedmont city, California                   10.119379   
367      San Ramon city, California                    7.562749   
3           Albany city, California                    8.673314   
262  Mountain View city, California                   10.334904   
141    Foster City city, California                    9.356232   

     attractiveness_score  
124             81.243140  
283             81.043293  
380             79.508606  
113             79.504302  
195             79.126269  
301             79.116006  
367             78.358474  
3               78.288300  
262       

## 5 Scenarios for Analysis

**Base Case :**
non_poverty_rate (35%), bachelors_or_higher_pct (25%), home_value_to_income_ratio (20%), labor_force_to_population_rate (20%)

**Equal Weights :** 
non_poverty_rate (25%), bachelors_or_higher_pct (25%), home_value_to_income_ratio (25%), labor_force_to_population_rate (25%)

**Pro-Affordability :**
non_poverty_rate (20%), bachelors_or_higher_pct (15%), home_value_to_income_ratio (15%), labor_force_to_population_rate (50%)

**Pro-Talent :**
non_poverty_rate (20%), bachelors_or_higher_pct (50%), home_value_to_income_ratio (15%), labor_force_to_population_rate (15%)

**Pro-Economy :**
non_poverty_rate (50%), bachelors_or_higher_pct (20%), home_value_to_income_ratio (15%), labor_force_to_population_rate (15%)

In [52]:
# Define test configurations
scenarios = {
    'Base Case': {'NP': 0.35, 'ED': 0.25, 'LF': 0.20, 'AFF': 0.20},
    'Equal Weights': {'NP': 0.25, 'ED': 0.25, 'LF': 0.25, 'AFF': 0.25},
    'Pro-Affordability': {'NP': 0.20, 'ED': 0.15, 'LF': 0.15, 'AFF': 0.50},
    'Pro-Talent': {'NP': 0.20, 'ED': 0.50, 'LF': 0.15, 'AFF': 0.15},
    'Pro-Economy': {'NP': 0.50, 'ED': 0.20, 'LF': 0.15, 'AFF': 0.15}
}

merged_zillow_ranks = pd.DataFrame(index=merged_zillow['place_name'])

# Compute rank arrays for comparison
for name, w in scenarios.items():
    score = (
        merged_zillow['non_poverty_rate'] * w['NP'] +
        merged_zillow['bachelors_or_higher_pct'] * w['ED'] +
        merged_zillow['labor_force_to_population_rate'] * w['LF'] +
        merged_zillow['affordability_score'] * w['AFF']
    ).values
    merged_zillow_ranks[name] = pd.Series(score).rank(ascending=False).astype(int).values

print("SENSITIVITY ANALYSIS (CITY RANKS):")
print(merged_zillow_ranks.head(10))

SENSITIVITY ANALYSIS (CITY RANKS):
                                  Base Case  Equal Weights  Pro-Affordability  \
place_name                                                                      
Adelanto city, California               410            413                310   
Agoura Hills city, California            15             17                 15   
Alameda city, California                 36             36                 64   
Albany city, California                   8              7                 29   
Alhambra city, California               227            234                342   
Aliso Viejo city, California             19             16                 12   
Alturas city, California                231            191                 19   
Amador City city, California             18             11                  2   
American Canyon city, California        162            143                 70   
Anaheim city, California                277            283                

In [53]:
NP = (merged_zillow_ranks["Base Case"].sort_values(ascending=True).head(10))
print(NP.head(10))

place_name
Emeryville city, California        1
Orinda city, California            2
Sausalito city, California         3
Dublin city, California            4
Lafayette city, California         5
Piedmont city, California          6
San Ramon city, California         7
Albany city, California            8
Mountain View city, California     9
Foster City city, California      10
Name: Base Case, dtype: int64


In [54]:
EQ = (merged_zillow_ranks["Equal Weights"].sort_values(ascending=True))
print(EQ.head(10))

place_name
Emeryville city, California        1
Orinda city, California            2
Sausalito city, California         3
Dublin city, California            4
Lafayette city, California         5
San Ramon city, California         6
Albany city, California            7
Piedmont city, California          8
Mountain View city, California     9
Foster City city, California      10
Name: Equal Weights, dtype: int64


In [55]:
PA = (merged_zillow_ranks["Pro-Affordability"].sort_values(ascending=True).head(10))
print(PA.head(10))

place_name
Emeryville city, California      1
Amador City city, California     2
Dublin city, California          3
Clayton city, California         4
Folsom city, California          5
Orinda city, California          6
Ridgecrest city, California      7
San Ramon city, California       8
Rocklin city, California         9
Hercules city, California       10
Name: Pro-Affordability, dtype: int64


In [56]:
PT = (merged_zillow_ranks["Pro-Talent"].sort_values(ascending=True).head(10))
print(PT.head(10))

place_name
Orinda city, California            1
Piedmont city, California          2
Lafayette city, California         3
Sausalito city, California         4
Emeryville city, California        5
Albany city, California            6
Cupertino city, California         7
Del Mar city, California           8
Los Altos city, California         9
Mountain View city, California    10
Name: Pro-Talent, dtype: int64


In [57]:
PE = (merged_zillow_ranks["Pro-Economy"].sort_values(ascending=True).head(10))
print(PE.head(10))

place_name
Orinda city, California            1
Lafayette city, California         2
Dublin city, California            3
Piedmont city, California          4
Sausalito city, California         5
San Ramon city, California         6
Emeryville city, California        7
Foster City city, California       8
Albany city, California            9
Mountain View city, California    10
Name: Pro-Economy, dtype: int64


In [58]:
# Define how many top cities you want to see per scenario
top_n = 10  

# Initialize an empty dictionary to compile our leaderboard
leaderboard_data = {}

# Loop through your calculated scenario configurations
for name, w in scenarios.items():
    # Calculate raw scores
    merged_zillow['temp_score'] = (
        merged_zillow['non_poverty_rate'] * w['NP'] +
        merged_zillow['bachelors_or_higher_pct'] * w['ED'] +
        merged_zillow['labor_force_to_population_rate'] * w['LF'] +
        merged_zillow['affordability_score'] * w['AFF']
    )
    
    # Sort the full dataframe by this temp score to keep alignment with 'place_name' column
    top_cities_df = merged_zillow.sort_values(by='temp_score', ascending=False).head(top_n)
    
    # Explicitly pull the column strings and the math values
    leaderboard_data[name] = [
        f"{row['place_name']} ({row['temp_score']:.1f})" for idx, row in top_cities_df.iterrows()
    ]

# Convert into a comparison DataFrame
leaderboard_df = pd.DataFrame(leaderboard_data)
leaderboard_df.index = [f"Rank {i+1}" for i in range(top_n)]

# Print the table
print(f"=== TOP {top_n} CITIES ACROSS ALL SCENARIOS ===")
print(leaderboard_df.to_string(line_width=160))

=== TOP 10 CITIES ACROSS ALL SCENARIOS ===
                                     Base Case                          Equal Weights                    Pro-Affordability  \
Rank 1      Emeryville city, California (81.2)     Emeryville city, California (81.0)   Emeryville city, California (86.7)   
Rank 2          Orinda city, California (81.0)         Orinda city, California (77.6)  Amador City city, California (82.0)   
Rank 3       Sausalito city, California (79.5)      Sausalito city, California (77.4)       Dublin city, California (81.9)   
Rank 4          Dublin city, California (79.5)         Dublin city, California (77.1)      Clayton city, California (79.9)   
Rank 5       Lafayette city, California (79.1)      Lafayette city, California (75.8)       Folsom city, California (79.5)   
Rank 6        Piedmont city, California (79.1)      San Ramon city, California (75.5)       Orinda city, California (79.4)   
Rank 7       San Ramon city, California (78.4)         Albany city, Califor